# LangChain POC

Notebook para aplicar Runnable y LCEL en dos modos: `mock` sin costo y `openai` con API real.

## Objetivos
- practicar `invoke`, `stream` y `batch`
- encadenar `prompt | model | parser` con LCEL
- probar salida estructurada
- poder seguir aunque la cuenta no tenga cuota en OpenAI API

In [3]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"

for path in (PROJECT_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SRC_DIR = {SRC_DIR}")

PROJECT_ROOT = c:\Users\andre\Desktop\4uentes\apps\4uentes-sst\chatboot-integration\sst_chatbot
SRC_DIR = c:\Users\andre\Desktop\4uentes\apps\4uentes-sst\chatboot-integration\sst_chatbot\src


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"

for path in (PROJECT_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

USE_OPENAI = True

from sst_chatbot.langchain_poc import (
    build_capital_chain,
    build_chat_model,
    build_mock_chat_model,
    build_structured_chat_model,
    build_topic_summary_chain,
)

if USE_OPENAI:
    from sst_chatbot.config import require_env

    require_env(("OPENAI_API_KEY", "LANGCHAIN_API_KEY"))
    chat_model = build_chat_model()
    print(f"Modo actual: openai | modelo: {chat_model.model_name}")
else:
    chat_model = build_mock_chat_model()
    print("Modo actual: mock | sin llamadas de red ni costo")

Modo actual: mock | sin llamadas de red ni costo


## 1. `invoke`

Tanto en modo `openai` como en modo `mock`, el objeto se usa como un runnable.

In [ ]:
from langchain_core.messages import HumanMessage

input_messages = [
    HumanMessage(content="Explica brevemente que es un Runnable en LangChain.")
]
response = chat_model.invoke(input_messages)
print(response.content)

## 2. `stream`

En `openai` vas a ver tokens reales. En `mock`, el stream emite una respuesta sintetica local.

In [ ]:
stream_messages = [
    HumanMessage(content="Escribi dos oraciones sobre LCEL y por que simplifica el encadenamiento.")
]

for chunk in chat_model.stream(stream_messages):
    if chunk.content:
        print(chunk.content, end="")

print()

## 3. LCEL con `prompt | model | parser`

Aca aplicamos el patron central de la clase.

In [ ]:
capital_chain = build_capital_chain(chat_model)
capital_chain.invoke({"pais": "Francia"})

## 4. `batch`

La misma cadena sigue siendo un runnable, asi que se puede ejecutar en lote.

In [ ]:
capital_chain.batch(
    [
        {"pais": "Argentina"},
        {"pais": "Uruguay"},
        {"pais": "Chile"},
    ]
)

## 5. Salida estructurada

Si el modelo soporta `with_structured_output`, se usa eso. Si estas en `mock`, devolvemos un objeto tipado local para practicar el flujo igual.

In [ ]:
structured_chain = build_topic_summary_chain(
    build_structured_chat_model(chat_model)
)

structured_result = structured_chain.invoke(
    {"topic": "LangChain Expression Language (LCEL)"}
)
structured_result.model_dump()

## Como se pensaron las pruebas

- `mock` sirve para pruebas manuales sin depender de billing.
- `openai` queda para validacion real cuando haya cuota disponible.
- Los tests unitarios validan composicion, prompts y tipado sin llamar a la API.
- La idea es separar aprendizaje del framework de la disponibilidad del proveedor.